In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.utils import load_dyst_samples

device = "cuda" if torch.cuda.is_available() else "cpu"
work_dir = os.getenv("WORK_DIR", "/work")
data_dir = os.path.join(work_dir, "data")

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "head_outputs")
os.makedirs(plot_save_dir, exist_ok=True)

In [ ]:
split_name = "base40"
subsample_interval = 1

split_dir = os.path.join(DATA_DIR, split_name)
system_name = "Lorenz"
# ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

context_length = 512
context_start_time = 2000
context_end_time = context_start_time + context_length

# Prepare input series
sample_idx = 0
selected_dim = 0
dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :]).unsqueeze(0)
dyst_coords = dyst_coords[:, ::subsample_interval]
context = dyst_coords[:, context_start_time:context_end_time]

print(context.shape)

prediction_length = 512
future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
print(f"future_vals shape: {future_vals.shape}")

X_train = context[0].numpy()
X_test = future_vals[0].numpy()
sol = context[0].T.numpy()

print(X_train.shape)
print(X_test.shape)

In [ ]:
from chronos import ChronosConfig, MeanScaleUniformBins
from sklearn.base import BaseEstimator, RegressorMixin


class NadarayaWatsonForecaster(BaseEstimator, RegressorMixin):
    """
    Nadaraya–Watson kernel regression forecaster with an autoregressive rollout.
    The prediction length sets the number of steps forecasted simultaneously, while
    the forecast length sets the total number of steps to forecast autoregressively.

    Attributes:
        n_neighbors (int): Number of nearest neighbors k.
        query_length (int): Length m of the lagged input window.
        prediction_length (int): Length h of each block forecast.
    """

    def __init__(self, n_neighbors=5, query_length=10, prediction_length=1):
        self.n_neighbors = n_neighbors
        self.query_length = query_length
        self.prediction_length = prediction_length

        n_tokens = 4096
        n_special_tokens = 2
        config = ChronosConfig(
            tokenizer_class="MeanScaleUniformBins",
            tokenizer_kwargs=dict(low_limit=-1.0, high_limit=1.0),
            n_tokens=n_tokens,
            n_special_tokens=n_special_tokens,
            pad_token_id=0,
            eos_token_id=1,
            use_eos_token=True,
            model_type="seq2seq",
            context_length=512,
            prediction_length=64,
            num_samples=20,
            temperature=1.0,
            top_k=50,
            top_p=1.0,
        )
        self.tokenizer = MeanScaleUniformBins(low_limit=-15.0, high_limit=15.0, config=config)

    def fit(self, series):
        series = np.asarray(series, dtype=float)
        N = series.shape[0]
        m, h = self.query_length, self.prediction_length
        T = N - m - h + 1
        if T <= 0:
            raise ValueError(f"Series length {N} too short for query_length={m} and prediction_length={h}")

        # build lagged inputs X (shape T×m) and targets Y (shape T×h)
        X = np.vstack([series[i : i + m] for i in range(T)])
        Y = np.vstack([series[i + m : i + m + h] for i in range(T)])

        self.X_ = X
        self.Y_ = Y
        _, _, self.scale = self.tokenizer.context_input_transform(torch.tensor(series).unsqueeze(0))
        return self

    def predict(self, initial_series, forecast_length, output_probs=False):
        """
        Roll out an h-step block predictor autoregressively to length H.

        Args:
            initial_series (array-like): Starting time series of length ≥ query_length.
            forecast_length (int): Total future steps H to forecast.
            output_probs (bool): Also outputs the probability of each y value in the forecast.

        Returns:
            array-like: Forecasted series of length H.
        """
        series = np.asarray(initial_series, dtype=float)
        m, h = self.query_length, self.prediction_length

        if series.shape[0] < m:
            raise ValueError(f"initial_series must have length ≥ query_length ({m}), got {series.shape[0]}")

        Q = series[-m:].copy()  # current window \mathbf{Q}\inℝ^m
        forecast = []

        probs = torch.zeros((forecast_length, 4096))

        while len(forecast) < forecast_length:
            # distances d_i = ||X_i − Q||₂
            dists = np.linalg.norm(self.X_ - Q, axis=1)
            # select k nearest
            idx = np.argsort(dists)[: self.n_neighbors]

            # adaptive bandwidth = d_(k)
            bandwidth = dists[idx[-1]] if dists[idx[-1]] > 0 else 1.0

            # Gaussian kernel weights w_i ∝ exp(−½(d_i/bandwidth)²)
            weights = np.exp(-0.5 * (dists[idx] / bandwidth) ** 2)
            weights /= weights.sum()

            # block forecast \mathbf{\hat{y}} = ∑ w_i Y_i
            y_block = weights.dot(self.Y_[idx])

            # Update probabilities for the tokens in y_token_ids
            if output_probs:
                # tokenize idx
                y_token_ids, _, _ = self.tokenizer._input_transform(
                    torch.tensor(self.Y_[idx]).unsqueeze(0), scale=self.scale
                )

                # Get the current position in the forecast
                current_pos = len(forecast)

                # Convert weights to tensor for easier operations
                weights_tensor = torch.tensor(weights)

                # y_token_ids has shape [1, n_neighbors, prediction_length]
                # We need to process each prediction step
                for step in range(min(h, forecast_length - current_pos)):
                    # Get tokens for this prediction step across all neighbors
                    step_tokens = y_token_ids[0, :, step]

                    # Create a dictionary to accumulate probabilities for tokens that appear multiple times
                    token_probs = {}

                    # Accumulate probabilities for each token
                    for i, token in enumerate(step_tokens):
                        token_id = token.item()
                        probs[current_pos + step, token_id] += weights_tensor[i].item()

            # truncate if remaining horizon < h
            remaining = forecast_length - len(forecast)
            if remaining < h:
                y_block = y_block[:remaining]

            forecast.extend(y_block.tolist())

            # autoregressive update: slide window to last m points
            Q = np.concatenate([Q, y_block])[-m:]

        if output_probs:
            return np.array(forecast), probs
        else:
            return np.array(forecast)

In [ ]:
## Test the Nadaraya-Watson forecast
# plt.figure(figsize=(10, 5))
# plt.plot(sol[:, 0], zorder=0, color='k')
# for query_length in [5]:
#     model = NadarayaWatsonForecaster(
#         n_neighbors=1,
#         query_length=query_length,
#         prediction_length=4
#     )
#     model.fit(X_train)
#     X_test_pred = model.predict(X_train, forecast_length=len(X_test))


#     # plt.plot(np.arange(len(X_train), len(X_train) + len(X_test)), X_test)
#     # plt.plot(np.arange(len(X_train), len(X_train) + len(X_test_pred)), X_test_pred, zorder=1)
#     plt.plot(
#         np.arange(len(X_train), len(X_train) + len(X_test_pred)),
#         X_test_pred,
#         zorder=1,
#         linewidth=2,
#         label=f"Query {model.query_length}, Neighbors {model.n_neighbors}"
#     )
# plt.legend(frameon=False)

n_neighbors = 4

plt.figure(figsize=(10, 5))
plt.plot(sol, zorder=0, color="k")

model = NadarayaWatsonForecaster(
    n_neighbors=n_neighbors,
    query_length=150,
    prediction_length=1,
)
model.fit(X_train)
X_test_pred, probs = model.predict(X_train, forecast_length=len(X_test), output_probs=True)

plt.plot(
    np.arange(len(X_train), len(X_train) + len(X_test_pred)),
    X_test_pred,
    zorder=1,
    linewidth=2,
    label=f"Query {model.query_length}, Neighbors {model.n_neighbors}",
)
plt.legend(frameon=False)

In [ ]:
# plot heatmap of probs

plt.figure(figsize=(10, 5))
# Use a different colormap with higher contrast
plt.imshow(probs.T[1500:2500, :], cmap="plasma", aspect="auto")
# Add normalization to enhance visibility of features
plt.clim(0, np.percentile(probs.T[1500:2500, :], 95))
# Add more descriptive colorbar and labels
cbar = plt.colorbar(label="Probability")
plt.xlabel("Time step")
plt.ylabel("Feature index")
plt.title("Probability Distribution Heatmap (Enhanced Contrast)")
plt.show()

In [ ]:
# Heatmap of kernel weights K(u_t, v) over growing context during autoregressive rollout
import matplotlib.pyplot as plt
import numpy as np

# H: total forecast horizon (number of steps predicted)
# L: initial context length
# m: query_length (window size)
# v: start index of each candidate window in the (growing) context
H = len(X_test_pred)
L = len(X_train)
m = model.query_length
k = model.n_neighbors

# Maximum number of candidate windows after forecasting H steps
max_num_windows = L + H - m + 1
K_heatmap = np.zeros((max_num_windows, H), dtype=float)

for t in range(H):
    # Context after t predicted steps
    S_t = np.concatenate([X_train, X_test_pred[:t]], axis=0)
    if S_t.shape[0] < m:
        continue

    # Query window: most recent m steps
    Q_t = S_t[-m:]

    # All candidate windows in S_t
    num_windows_t = S_t.shape[0] - m + 1
    X_t = np.vstack([S_t[i : i + m] for i in range(num_windows_t)])

    # Euclidean distances to the query window
    dists = np.linalg.norm(X_t - Q_t[None, :], axis=1)

    # Bandwidth = distance to k-th nearest neighbor (fallback to 1.0 if zero)
    idx_sorted = np.argsort(dists)
    kth = min(k, len(idx_sorted)) - 1
    bandwidth = dists[idx_sorted[kth]]
    if bandwidth <= 0:
        bandwidth = 1.0

    # Gaussian kernel and per-column normalization
    weights = np.exp(-0.5 * (dists / bandwidth) ** 2)
    s = weights.sum()
    if s > 0:
        weights = weights / s

    # Place into heatmap (pad remaining rows with zeros)
    K_heatmap[:num_windows_t, t] = weights

plt.figure(figsize=(10, 6))
im = plt.imshow(K_heatmap, aspect="auto", origin="lower", cmap="viridis")
plt.colorbar(im, label="K(u_t, v)")
plt.xlabel("Forecast step t (0-indexed)")
plt.ylabel("Start index v")
plt.title(f"Kernel weights over growing context (H={H}, L={L}, m={m})")
plt.tight_layout()
